In [3]:
from ML_pipeline.data_preprocessing import HeartbeatDataProcessor 

In [4]:
folder_path='../data/PAMAP2_Dataset/Protocol/'
filtered_df_path='../ML_pipeline/'
processed_data = HeartbeatDataProcessor(folder_path,filtered_df_path)
processed_data.preprocess_subjects(range(101,108))
# print(processed_data.df_filtered.head(3))
# print(processed_data.filtered_index.head(3))

Selected DataFrame columns have 0.2913% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.019% NaNs!

Filtering 51 columns (excluding 0, 1, 2)...
successfully loaded subject 101
Selected DataFrame columns have 0.4147% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0327% NaNs!

Filtering 51 columns (excluding 0, 1, 2)...
successfully loaded subject 102
Selected DataFrame columns have 0.1624% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0117% NaNs!

Filtering 51 columns (excluding 0, 1, 2)...
successfully loaded subject 103
Selected DataFrame columns have 0.3568% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0095% NaNs!

Filtering 51 columns (excluding 0, 1, 2)...
successfully loaded subject 104
Selected DataFrame columns have 0.3409% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0268% NaNs!

Filtering 51 columns (excluding 0, 1, 2)..

In [5]:
print(processed_data.df_filtered.head(3))

       0  1   2        3         4         5         6         7         8  \
0  89.00  1 NaN  30.1875 -9.504414  2.424780  0.469282 -9.511102  2.485077   
1  89.01  1 NaN  30.1875 -9.473263  2.407320  0.451058 -9.505700  2.470919   
2  89.02  1 NaN  30.1875 -9.448844  2.390175  0.435815 -9.502204  2.466053   

          9  ...        46         47         48        49        50  \
0  0.541734  ...  0.007468 -51.433089 -45.757012 -2.066981  0.483099   
1  0.526272  ...  0.006749 -51.516419 -45.753806 -2.042584  0.483001   
2  0.515158  ...  0.005491 -51.542121 -45.729977 -2.014109  0.482940   

         51        52        53  subject_id  interval_id  
0 -0.458459  0.660026 -0.347558         107           78  
1 -0.458507  0.660000 -0.347671         107           78  
2 -0.458523  0.659990 -0.347738         107           78  

[3 rows x 56 columns]


## Random Forest on subject 101 segments

This builds per-window statistical features from `processed_data.subject_segment_dict[101]`, then trains and evaluates a Random Forest classifier.

In [7]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

slices = []
for i in range(101,108):
    slices.extend([s for s in processed_data.subject_segment_dict[i] if not s.empty])

feature_rows = []
labels = []
groups = []

for slice in slices:
    if slice.empty:
        continue

    # Column 1 is activity id in PAMAP2 raw files.
    label = int(slice[1].mode().iloc[0])

    # Keep numeric sensor columns, excluding metadata/label columns.
    drop_cols = [0, 1,2, 7,8,9,16,17,18,19,24,25,26,33,34,35,36,41,42,43,50,51,52,53, "subject_id", "interval_id"]
    sensor_df = slice.drop(columns=drop_cols, errors="ignore")
    sensor_df = sensor_df.select_dtypes(include=[np.number])

    if sensor_df.shape[1] == 0:
        continue

    row = {}
    for col in sensor_df.columns:
        values = sensor_df[col]
        row[f"c{col}_mean"] = values.mean()
        row[f"c{col}_std"] = values.std()
        row[f"c{col}_min"] = values.min()
        row[f"c{col}_max"] = values.max()
        row[f"c{col}_median"] = values.median()
        row[f"c{col}_PtP_amplitude"] = values.max() - values.min()
        row[f"c{col}_harmonic_mean"] = 1 / np.mean(1 / values)
        row[f"c{col}_interquartile_range"] = values.quantile(0.75) - values.quantile(0.25)
        row[f"c{col}_kurtosis"] = values.kurtosis()
        row[f"c{col}_skewness"] = values.skew()
        # fft_values = pd.Series(np.fft.fft(values))
        fft_vals = np.fft.rfft(values.to_numpy())
        fft_mag = np.abs(fft_vals)
        row[f"c{col}_fft_mean"] = fft_mag.mean()
        row[f"c{col}_fft_std"] = fft_mag.std()
        row[f"c{col}_fft_min"] = fft_mag.min()
        row[f"c{col}_fft_max"] = fft_mag.max()
        # row[f"c{col}_fft_median"] = fft_mag.median()



    feature_rows.append(row)
    labels.append(label)
    groups.append(int(slice["interval_id"].iloc[0]))


In [ ]:


X = pd.DataFrame(feature_rows)
y = pd.Series(labels, name="activity_id")
groups = pd.Series(groups, name="interval_id")

# print(f"X shape: {X.shape}")
print("Class counts:")
print(y.value_counts().sort_index())

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf_pipeline = Pipeline([
    # ("imputer", SimpleImputer(strategy="median")),
    (
        "rf",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            max_depth=24,
            n_jobs=-1,
            criterion="gini",
            class_weight="balanced_subsample",
        ),
    ),
])

rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels_sorted = np.sort(np.unique(np.concatenate([y_test.to_numpy(), y_pred])))
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels_sorted),
    index=[f"true_{lbl}" for lbl in labels_sorted],
    columns=[f"pred_{lbl}" for lbl in labels_sorted],
)

print("\nConfusion matrix (test set):")
#print(cm)

Class counts:
activity_id
1     1599
2     1539
3     1563
4     1992
5      756
6     1320
7     1521
12     887
13     771
16    1426
17    1973
24     302
Name: count, dtype: int64

Accuracy: 0.8209

Classification report:
              precision    recall  f1-score   support

           1       1.00      0.91      0.95       686
           2       0.88      0.98      0.93       462
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           5       1.00      0.98      0.99       281
           6       0.97      0.60      0.74       234
           7       1.00      0.68      0.81       729
          12       0.93      0.98      0.96       198
          13       0.92      1.00      0.96        99
          16       0.85      0.71      0.77       645
          17       0.00      0.00      0.00         0

    accuracy                           0.82      3334
   macro avg       0.69      0.62      0.65      3334
weighted avg    

1 : lying
2 : sitting

4 : walking
7 : Nodic walking

6 : cycling
16: vaccum cleaning
17: ironing